# 🕵️ Dia 37 — O que contém um perfil de ator de ameaça

**Investigador Cyber 90 Dias** · Fase 3 · Rank Jutsu `B`

> Baseado no *Day 37* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-3-cyber-threat-intelligence/day-37.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

A Marta recebeu três nomes para o mesmo golpe: "PixLure" (SOC da CoopVale), "Garça Azul" (um *post* de fornecedor que fala em "quadrilha ligada a facção") e "Pescador" (a Vigia). Você começou a escrever *"Perfil do Pescador"*. Rafa apagou: *"Pescador é uma string num JavaScript. O que você perfila é um **cluster de atividade**: CA-ISCA-01."*

Neste notebook você vai:

1. **Comparar os aliases do APT29** no ATT&CK e no MISP galaxy, e achar as discordâncias do original por código.
2. **Montar o crosswalk do CA-ISCA-01** e marcar as falsas equivalências.
3. **Jogar o quiz do Admiralty.**
4. **Gerar o perfil em Markdown**, com um verificador que recusa frase sem fonte e data.
5. **Desenhar o Diamante** do evento de 5/out.

🐍 Só biblioteca padrão do Python. Nada para instalar. Funciona offline: toda célula que tenta a rede tem plano B embutido. Rode as células de cima para baixo (Shift+Enter).

## 1. Aliases do APT29: ATT&CK × MISP galaxy (passo 1 do original)

A célula tenta baixar o `threat-actor.json` do MISP galaxy. Sem rede, usa um plano B com as três entradas relevantes (APT29, UNC2452, APT28), **conferidas em 2026-10-12**. Os aliases do ATT&CK (G0016, v19.2) estão embutidos.

Repare na função `quem_cita()`: ela compara **sem diferenciar maiúsculas** (`casefold`), ao contrário do `index()` do jq. Isso muda o resultado.

In [ ]:
import json            # o MISP galaxy é um JSON
import urllib.request  # para baixar, sem instalar nada

# Aliases do APT29 no ATT&CK Enterprise v19.2 (campo "aliases" do intrusion-set G0016)
ATTCK_APT29 = ['APT29', 'IRON RITUAL', 'IRON HEMLOCK', 'NobleBaron', 'Dark Halo', 'NOBELIUM', 'UNC2452', 'YTTRIUM', 'The Dukes', 'Cozy Bear', 'CozyDuke', 'SolarStorm', 'Blue Kitsune', 'UNC3524', 'Midnight Blizzard']

# Plano B: as três entradas do MISP que interessam hoje, conferidas em 2026-10-12
MISP_PLANO_B = {"APT29": ["Group 100", "COZY BEAR", "The Dukes", "Minidionis", "SeaDuke", "YTTRIUM", "IRON HEMLOCK", "Grizzly Steppe", "G0016", "ATK7", "Cloaked Ursa", "TA421", "Blue Kitsune", "ITG11", "BlueBravo", "Nobelium", "UAC-0029", "ICECAP", "ICE RELIC"], "UNC2452": ["DarkHalo", "StellarParticle", "NOBELIUM", "Solar Phoenix", "Midnight Blizzard"], "APT28": ["Pawn Storm", "FANCY BEAR", "Sednit", "SNAKEMACKEREL", "Tsar Team", "TG-4127", "STRONTIUM", "Swallowtail", "IRON TWILIGHT", "Group 74", "SIG40", "Grizzly Steppe", "G0007", "ATK5", "Fighting Ursa", "ITG05", "Blue Athena", "TA422", "T-APT-12", "APT-C-20", "UAC-0028", "UAC-0001", "FROZENLAKE", "Sofacy", "Forest Blizzard", "BlueDelta", "Fancy Bear", "GruesomeLarch", "LAKE RELIC"]}

URL_MISP = "https://raw.githubusercontent.com/MISP/misp-galaxy/main/clusters/threat-actor.json"
try:
    with urllib.request.urlopen(URL_MISP, timeout=30) as r:
        valores = json.load(r)["values"]
    # transformamos em {nome_da_entrada: [sinônimos]} — o mesmo formato do plano B
    MISP = {v["value"]: (v.get("meta", {}).get("synonyms") or []) for v in valores}
    origem = f"MISP galaxy pela rede ({len(MISP)} entradas)"
except Exception as erro:
    MISP = MISP_PLANO_B
    origem = f"plano B embutido (3 entradas, 2026-10-12) — {erro.__class__.__name__}"
print("Fonte MISP:", origem)

def quem_cita(nome):
    """Lista as entradas do MISP cujo nome ou sinônimos contêm `nome` (sem diferenciar maiúsculas)."""
    alvo = nome.casefold()
    return [entrada for entrada, sinonimos in MISP.items()
            if alvo == entrada.casefold() or alvo in (s.casefold() for s in sinonimos)]

print("\nATT&CK G0016:", "; ".join(ATTCK_APT29))
for nome in ["Midnight Blizzard", "NOBELIUM", "UNC2452", "Grizzly Steppe", "Cozy Bear"]:
    no_attck = nome.casefold() in (a.casefold() for a in ATTCK_APT29)
    print(f"{nome:<18} ATT&CK→APT29? {'sim' if no_attck else 'não':<4} | MISP: {quem_cita(nome)}")

> 💡 **Leitura:** o ATT&CK põe Midnight Blizzard, NOBELIUM e UNC2452 sob o APT29; o MISP mantém uma entrada `UNC2452` separada com Midnight Blizzard. E "Grizzly Steppe" aparece sob APT28 **e** APT29 no MISP: é o título de um relatório de 2016, não um nome de ator. Dois catálogos sérios, duas respostas. É por isso que o crosswalk precisa da coluna **"afirmado por"**.

## 2. ➕ O caso: fontes, nomes e afirmações sobre o CA-ISCA-01

Embutido abaixo, igual ao arquivo `caso-continuo/dados/dia-37/fontes-ca-isca-01.json` do repositório. Tudo fictício.

In [ ]:
CASO = {
 "_aviso": "Dados 100% fictícios da Operação Isca (Investigador Cyber 90 Dias, Dia 37). Organizações, nomes de cluster e relatórios são inventados. Domínios .example (RFC 2606). Nenhuma pessoa real. O cluster CA-ISCA-01 é um agrupamento de atividade, não uma pessoa.",
 "cluster": "CA-ISCA-01",
 "fontes": [
  {
   "id": "F1",
   "titulo": "Relatório interno VIGIA-2026-001 (triagem do SMS 'Pix bloqueado')",
   "autor": "Vigia",
   "tipo": "análise própria com evidência primária (SMS, capturas com hash, crt.sh)",
   "publicado": "2026-10-06"
  },
  {
   "id": "F2",
   "titulo": "Alerta conjunto ALR-2026-01",
   "autor": "Vigia + grupo de segurança das cooperativas mineiras",
   "tipo": "alerta conjunto de várias organizações",
   "publicado": "2026-10-12"
  },
  {
   "id": "F3",
   "titulo": "Nota interna do SOC da CoopVale sobre 'PixLure'",
   "autor": "SOC CoopVale (fictício)",
   "tipo": "nota de outro time, telemetria própria",
   "publicado": "2026-10-09"
  },
  {
   "id": "F4",
   "titulo": "Post de blog 'Garça Azul: o kit que pesca cooperativas'",
   "autor": "Fornecedor de threat intel fictício 'Radar Sul'",
   "tipo": "blog de fornecedor, sem IOCs publicados",
   "publicado": "2026-10-10"
  },
  {
   "id": "F5",
   "titulo": "Anúncio do IscaKit num grupo de mensagens (print recebido por um cooperado)",
   "autor": "desconhecido",
   "tipo": "print de terceiro, sem cadeia de custódia",
   "publicado": "2026-10-08"
  },
  {
   "id": "F6",
   "titulo": "Reportagem 'Golpe do Pix bloqueado volta a assustar' em portal de notícias fictício",
   "autor": "Portal fictício",
   "tipo": "imprensa, fala do golpe em geral",
   "publicado": "2026-10-07"
  }
 ],
 "nomes": [
  {
   "nome": "CA-ISCA-01",
   "organizacao": "Vigia",
   "afirmado_equivalente_por": "—",
   "nota": "Nome de trabalho do cluster (Dia 43)."
  },
  {
   "nome": "Pescador",
   "organizacao": "Vigia (apelido interno)",
   "afirmado_equivalente_por": "F1",
   "nota": "Vem da string '// pescador' no JS do kit. É apelido de artefato, NÃO de pessoa."
  },
  {
   "nome": "PixLure",
   "organizacao": "SOC CoopVale",
   "afirmado_equivalente_por": "F3",
   "nota": "Telemetria da CoopVale; cobre só os SMS que chegaram aos cooperados dela."
  },
  {
   "nome": "Garça Azul",
   "organizacao": "Radar Sul (fornecedor fictício)",
   "afirmado_equivalente_por": "F4",
   "nota": "O post junta o IscaKit com uma campanha de 2025 contra bancos digitais. Escopo maior que o nosso."
  },
  {
   "nome": "IscaKit",
   "organizacao": "Vigia",
   "afirmado_equivalente_por": "F1",
   "nota": "É o nome do KIT (ferramenta), não do cluster. Ferramenta vendida: outros operadores podem usar."
  },
  {
   "nome": "Golpe do Pix bloqueado",
   "organizacao": "imprensa (F6)",
   "afirmado_equivalente_por": "F6",
   "nota": "Nome de um TIPO de golpe usado por muitos grupos sem relação. Não é alias do cluster."
  }
 ],
 "afirmacoes": [
  {
   "id": "A1",
   "campo": "targeting",
   "texto": "Cooperativas de crédito de Minas Gerais, via SMS a cooperados",
   "fonte": "F2",
   "data_atividade": "2026-10-05/2026-10-11"
  },
  {
   "id": "A2",
   "campo": "ttp",
   "texto": "Registro de domínios imitação 1 a 10 dias antes do disparo (T1583.001)",
   "fonte": "F1",
   "data_atividade": "2026-10-02/2026-10-10"
  },
  {
   "id": "A3",
   "campo": "ttp",
   "texto": "Certificado TLS gratuito obtido na véspera do disparo (T1588.004)",
   "fonte": "F1",
   "data_atividade": "2026-10-04"
  },
  {
   "id": "A4",
   "campo": "ttp",
   "texto": "SMS com link encurtado para o domínio imitação (T1660 Mobile; T1566.002 Enterprise)",
   "fonte": "F2",
   "data_atividade": "2026-10-05"
  },
  {
   "id": "A5",
   "campo": "ttp",
   "texto": "Captura da senha na página falsa (T1056.003) e Pix com credenciais válidas (T1078, T1657)",
   "fonte": "F2",
   "data_atividade": "2026-10-05"
  },
  {
   "id": "A6",
   "campo": "tooling",
   "texto": "IscaKit, kit de phishing anunciado à venda em grupo de mensagens",
   "fonte": "F5",
   "data_atividade": "2026-10-08"
  },
  {
   "id": "A7",
   "campo": "tooling",
   "texto": "Encurtador público lnk.example",
   "fonte": "F1",
   "data_atividade": "2026-10-05"
  },
  {
   "id": "A8",
   "campo": "infra",
   "texto": "Hospedagem em 203.0.113.77; NS ns1/ns2.dnsbarato.example; certificados com vários nomes no SAN",
   "fonte": "F1",
   "data_atividade": "2026-10-04/2026-10-09"
  },
  {
   "id": "A9",
   "campo": "motivacao",
   "texto": "Financeira (transferências Pix não reconhecidas)",
   "fonte": "F2",
   "data_atividade": "2026-10-05"
  },
  {
   "id": "A10",
   "campo": "sponsor",
   "texto": "Grupo criminoso do Sudeste com ligação a facção",
   "fonte": "F4",
   "data_atividade": "não informada"
  }
 ]
}

print(f"{len(CASO['fontes'])} fontes, {len(CASO['nomes'])} nomes, {len(CASO['afirmacoes'])} afirmações.")
for f in CASO["fontes"]:
    print(f"  {f['id']}: {f['titulo']} — {f['tipo']}")

## 3. Crosswalk do CA-ISCA-01

Para cada nome, você decide a **classe**: `cluster` (nome de agrupamento de atividade), `ferramenta`, `tipo-de-golpe` ou `artefato`. Só `cluster` pode ser candidato a alias. E mesmo entre clusters, escopo diferente não é equivalência exata.

Edite `CLASSE` se discordar e rode.

In [ ]:
# Sua classificação de cada nome. EDITE se discordar.
CLASSE = {
    "CA-ISCA-01": "cluster",
    "Pescador": "artefato",              # string '// pescador' no JS: não é pessoa, não é cluster
    "PixLure": "cluster",                # nome do SOC da CoopVale, telemetria só dela
    "Garça Azul": "cluster",             # nome do fornecedor, mas com escopo maior (inclui 2025)
    "IscaKit": "ferramenta",             # kit vendido: outros operadores podem usar
    "Golpe do Pix bloqueado": "tipo-de-golpe",
}
SINALIZAR = {"ferramenta": "⚠️ nome de FERRAMENTA, não de cluster",
             "tipo-de-golpe": "⛔ nome de TIPO de golpe (o 'Grizzly Steppe' daqui): não adote",
             "artefato": "🔎 apelido de ARTEFATO: registre como evidência, nunca como identidade"}

linhas = ["| Nome | Organização | Afirmado equivalente por | Classe | Nota de escopo |", "|---|---|---|---|---|"]
for n in CASO["nomes"]:
    classe = CLASSE.get(n["nome"], "?")
    nota = SINALIZAR.get(classe, n["nota"])
    if classe == "cluster" and n["nome"] != "CA-ISCA-01":
        nota = "≈ sobreposição, não equivalência exata. " + n["nota"]
    linhas.append(f"| {n['nome']} | {n['organizacao']} | {n['afirmado_equivalente_por']} | {classe} | {nota} |")
CROSSWALK_MD = "\n".join(linhas)
print(CROSSWALK_MD)

## 4. 🎮 Quiz do Admiralty

Para cada fonte, escolha a **letra de confiabilidade** (A a F). O gabarito do Rafa vem com o porquê. Não existe nota "certa" absoluta: existe nota **justificada**. O notebook aceita uma letra de distância como acerto parcial.

| Letra | Confiabilidade da fonte | Número | Credibilidade da informação |
|---|---|---|---|
| A | totalmente confiável | 1 | confirmada por outras fontes |
| B | normalmente confiável | 2 | provavelmente verdadeira |
| C | razoavelmente confiável | 3 | possivelmente verdadeira |
| D | normalmente não confiável | 4 | duvidosa |
| E | não confiável | 5 | improvável |
| F | não pode ser avaliada | 6 | não pode ser avaliada |

In [ ]:
MODO_JOGO = True   # False = só mostra o gabarito

GABARITO = {   # fonte: (letra, número típico das afirmações dela, por quê)
    "F1": ("B", 2, "Análise própria com evidência primária e hash. Não é A: é a primeira semana do caso e é a nossa."),
    "F2": ("B", 2, "Várias organizações revisaram; mas parte vem da F1 (não é independente dela)."),
    "F3": ("C", 3, "Outro time, telemetria própria, mas só da CoopVale e sem ver os dados dele."),
    "F4": ("F", 6, "Fornecedor sem histórico conosco e sem IOCs: não dá para avaliar. A afirmação de 'facção' não tem evidência."),
    "F5": ("F", 3, "Print de terceiro sem custódia: não dá para avaliar a fonte; a informação (o kit é vendido) é plausível."),
    "F6": ("C", 4, "Imprensa fala do tipo de golpe em geral; útil para contexto, fraca para o cluster."),
}
LETRAS = "ABCDEF"

pontos = 0.0
for f in CASO["fontes"]:
    letra, num, porque = GABARITO[f["id"]]
    print(f"\n{f['id']}: {f['titulo']}  ({f['tipo']})")
    if MODO_JOGO:
        r = ""
        while r not in LETRAS or r == "":
            r = input("   Confiabilidade (A-F): ").strip().upper()[:1]
        dist = abs(LETRAS.index(r) - LETRAS.index(letra))
        pontos += 1 if dist == 0 else (0.5 if dist == 1 else 0)
        print("   ✅" if dist == 0 else ("   🟡 quase" if dist == 1 else "   ❌"), end=" ")
    print(f"Rafa: {letra}{num} — {porque}")
if MODO_JOGO:
    print(f"\n🏁 {pontos}/{len(CASO['fontes'])}")

## 5. O perfil, com verificador

O checkpoint do original diz: *qualquer frase do perfil nomeia o documento de onde veio e a data da atividade; corte a que não fizer isso.* A função `verificar()` faz exatamente isso antes de escrever o arquivo. Ela também recusa:
- o nome "Pescador" usado como sujeito de uma frase (perfil de pessoa disfarçado);
- a afirmação de afiliação (A10) no resumo.

In [ ]:
import re   # para achar as marcas [Fx, data] nas frases

NOTA = {fid: f"{l}{n}" for fid, (l, n, _) in GABARITO.items()}     # F1 -> "B2"
AF = {a["id"]: a for a in CASO["afirmacoes"]}

def frase(aid):
    """Monta uma frase do perfil a partir de uma afirmação, já com fonte, nota e data."""
    a = AF[aid]
    return f"{a['texto']} [{a['fonte']} ({NOTA[a['fonte']]}), atividade: {a['data_atividade']}]"

RESUMO = [
    "Cluster de atividade de phishing por SMS contra cooperativas de crédito de MG, ativo em out/2026 "
    "[F2 (B2), atividade: 2026-10-05/2026-10-11]",
    "Usa um kit comercial (IscaKit) com domínios imitação por cooperativa e certificados obtidos na véspera "
    "[F1 (B2), atividade: 2026-10-02/2026-10-10]",
]

def verificar(linhas, onde):
    """Devolve a lista de problemas. Lista vazia = aprovado."""
    problemas = []
    for l in linhas:
        if not re.search(r"\[F\d \([A-F]\d\), atividade: [^\]]+\]", l):
            problemas.append(f"{onde}: sem fonte/data → {l[:60]}")
        if re.match(r"\s*(o\s+)?pescador\b", l, re.I):
            problemas.append(f"{onde}: 'Pescador' como sujeito (perfil de pessoa?) → {l[:60]}")
        if onde == "resumo" and ("facção" in l.lower() or "F4" in l):
            problemas.append(f"{onde}: afiliação não confirmada no resumo → {l[:60]}")
    return problemas

# Teste do verificador com duas frases ruins de propósito
for ruim in ["O Pescador é um criminoso experiente de BH.", frase("A10")]:
    print("Teste:", verificar([ruim], "resumo") or "aprovada")

In [ ]:
def secao(campo):
    return [frase(a["id"]) for a in CASO["afirmacoes"] if a["campo"] == campo]

problemas = verificar(RESUMO, "resumo")
for campo in ["targeting", "ttp", "tooling", "infra", "motivacao", "sponsor"]:
    problemas += verificar(secao(campo), campo)

if problemas:
    print("⛔ Perfil recusado:"); print("\n".join(problemas))
else:
    perfil = f"""# Perfil de cluster de atividade: CA-ISCA-01
Versão do ATT&CK: Enterprise e Mobile v19.2 | Data do perfil: 2026-10-12 | Analista: você (Vigia) | TLP:AMBER

> "Pescador" é o apelido de um ARTEFATO (a string `// pescador` no JavaScript do kit). Não designa pessoa.

## Resumo
{chr(10).join('- ' + l for l in RESUMO)}

## Nomes → crosswalk
{CROSSWALK_MD}

## Patrocinador ou afiliação suspeitos
{chr(10).join('- ' + l for l in secao('sponsor'))}
- Status: NÃO confirmado. Faltaria: fonte primária com evidência (ex.: relatório policial público, decisão judicial) e corroboração independente.

## Motivação
{chr(10).join('- ' + l for l in secao('motivacao'))}

## Alvos (com datas)
{chr(10).join('- ' + l for l in secao('targeting'))}

## TTPs (com datas, IDs do ATT&CK v19.2)
{chr(10).join('- ' + l for l in secao('ttp'))}

## Ferramental
| Ferramenta | Exclusiva ou compartilhada? | Fonte |
|---|---|---|
| IscaKit | COMPARTILHADA / comercial (anunciado à venda): não identifica o cluster | F5 (F3), F1 (B2) |
| Encurtador lnk.example | COMPARTILHADA (serviço público) | F1 (B2) |

## Padrões de infraestrutura
{chr(10).join('- ' + l for l in secao('infra'))}

## Perguntas em aberto
- PixLure (CoopVale) e CA-ISCA-01 cobrem a mesma atividade? Falta ver a telemetria da F3.
- A campanha de 2025 citada pela F4 tem relação? Nenhuma fonte primária nossa a viu.

## Fontes (Admiralty)
{chr(10).join(f"- {f['id']} {NOTA[f['id']]}: {f['titulo']} ({f['autor']}, {f['publicado']}) — {GABARITO[f['id']][2]}" for f in CASO['fontes'])}
"""
    with open("perfil-ca-isca-01.md", "w", encoding="utf-8") as fh:
        fh.write(perfil)
    print(perfil)
    print("💾 perfil-ca-isca-01.md salvo.")

## 6. O Diamante do evento de 5/out/2026 (passo 4 do original, versão Isca)

Cada vértice: afirmação + fonte. Onde a fonte não diz, escreva **"não informado na fonte"**, mesmo que você "saiba" por outro caminho.

In [ ]:
#@title Diamante — evento de 2026-10-05 na Serra Azul { display-mode: "form" }
adversario = "CA-ISCA-01 (nome de trabalho da Vigia) [F1]"  #@param {type:"string"}
capacidade = "IscaKit: página falsa + captura de senha; SMS com link [F1, F2]"  #@param {type:"string"}
infraestrutura = "serraazul-pix.example em 203.0.113.77; lnk.example/sa7 [F1]. Origem dos SMS: não informado na fonte"  #@param {type:"string"}
vitima = "Cooperados da Serra Azul (37 reclamações; 3 com Pix não reconhecido) [F2]"  #@param {type:"string"}

import textwrap   # quebra texto longo em linhas sem cortar palavras

def bloco(rotulo, conteudo, recuo):
    # Rótulo + conteúdo quebrado em linhas de 40 caracteres, com recuo.
    linhas = [rotulo] + textwrap.wrap(conteudo, 40)
    return "\n".join(" " * recuo + l for l in linhas)

texto = "\n".join([
    bloco("ADVERSÁRIO", adversario, 18), " " * 16 + "/          \\",
    bloco("CAPACIDADE", capacidade, 2), bloco("INFRAESTRUTURA", infraestrutura, 34),
    " " * 16 + "\\          /", bloco("VÍTIMA", vitima, 18)])
print(texto)

mermaid = f"""```mermaid
graph TD
    ADV["ADVERSÁRIO<br/>{adversario}"]
    CAP["CAPACIDADE<br/>{capacidade}"]
    INF["INFRAESTRUTURA<br/>{infraestrutura}"]
    VIC["VÍTIMA<br/>{vitima}"]
    ADV --- CAP
    ADV --- INF
    CAP --- VIC
    INF --- VIC
    CAP --- INF
```"""
with open("diamante-2026-10-05.md", "w", encoding="utf-8") as fh:
    fh.write("# Diamante — 2026-10-05, Serra Azul\n\n" + mermaid + "\n")
print("💾 diamante-2026-10-05.md salvo (cole no GitHub para ver o desenho).")

try:
    from google.colab import files
    for nome in ["perfil-ca-isca-01.md", "diamante-2026-10-05.md"]:
        files.download(nome)
except ImportError:
    print("(Fora do Colab: os arquivos ficaram na pasta atual.)")

## ✅ Antes de fechar o Dia 37

- [ ] Crosswalk do APT29 com 8+ nomes e o "Grizzly Steppe" sinalizado
- [ ] Crosswalk do CA-ISCA-01 com IscaKit (ferramenta) e "Golpe do Pix bloqueado" (tipo de golpe) sinalizados
- [ ] `perfil-ca-isca-01.md` aprovado pelo verificador
- [ ] A10 fora do resumo, com nota baixa e o que faltaria
- [ ] Diamante com pelo menos um "não informado na fonte"

**Amanhã (Dia 38):** o que é **distintivo** num cluster. Uma string no JavaScript vale mais ou menos que um IP? A Pirâmide da Dor responde.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>